<a href="https://colab.research.google.com/github/tarasovEgor/DeepLearningKurs/blob/main/src/lab_1/cnn_clf_lab_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Лабораторная работа 1.**

---

## Классификация изображений с использованием CNN, предобработка, аугментация и анализ метрик.

---

## **Цель работы:** изучить основы сверточных нейронных сетей (CNN) в PyTorch, исследовать влияние предобработки, аугментации, архитектуры сети и гиперпараметров на качество классификации изображений.

### 0. Init steps

---

#### Импорты библиотек:

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torch.optim.lr_scheduler import CyclicLR

from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Устройство:", device)

Устройство: cuda


#### Вспомогательные функции:

---

##### 0.1 Загрузка данных:

In [4]:
def get_transforms(dataset_name, augment=False):
    if dataset_name == "CIFAR10":
        if augment:
            train_transform = transforms.Compose([
                transforms.RandomHorizontalFlip(),
                transforms.RandomCrop(32, padding=4),
                transforms.ToTensor(),
                transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
            ])
        else:
            train_transform = transforms.Compose([
                transforms.ToTensor(),
                transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
            ])

        test_transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
        ])

    elif dataset_name == "FashionMNIST":
        if augment:
            train_transform = transforms.Compose([
                transforms.RandomRotation(10),
                transforms.ToTensor(),
                transforms.Normalize((0.5,), (0.5,))
            ])
        else:
            train_transform = transforms.Compose([
                transforms.ToTensor(),
                transforms.Normalize((0.5,), (0.5,))
            ])

        test_transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.5,), (0.5,))
        ])

    elif dataset_name == "SVHN":
        if augment:
            train_transform = transforms.Compose([
                transforms.ColorJitter(brightness=0.2),
                transforms.RandomRotation(10),
                transforms.ToTensor(),
                transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
            ])
        else:
            train_transform = transforms.Compose([
                transforms.ToTensor(),
                transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
            ])

        test_transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
        ])

    return train_transform, test_transform


def get_dataloaders(dataset_name, batch_size=64, augment=False):
    train_transform, test_transform = get_transforms(dataset_name, augment)

    if dataset_name == "CIFAR10":
        train_data = datasets.CIFAR10(root="./data", train=True, download=True, transform=train_transform)
        test_data = datasets.CIFAR10(root="./data", train=False, download=True, transform=test_transform)
        in_channels = 3
        img_size = 32
        num_classes = 10

    elif dataset_name == "FashionMNIST":
        train_data = datasets.FashionMNIST(root="./data", train=True, download=True, transform=train_transform)
        test_data = datasets.FashionMNIST(root="./data", train=False, download=True, transform=test_transform)
        in_channels = 1
        img_size = 28
        num_classes = 10

    elif dataset_name == "SVHN":
        train_data = datasets.SVHN(root="./data", split="train", download=True, transform=train_transform)
        test_data = datasets.SVHN(root="./data", split="test", download=True, transform=test_transform)
        in_channels = 3
        img_size = 32
        num_classes = 10

    train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False)

    return train_loader, test_loader, in_channels, img_size, num_classes

##### 0.2 Базовая CNN:

In [5]:
class BasicCNN(nn.Module):
    def __init__(
        self,
        in_channels=3,
        num_classes=10,
        img_size=32,
        filters=(32, 64),
        kernel_sizes=(3, 3),
        use_bn=False,
        dropout=0.0,
        pool_type="max"
    ):
        super().__init__()

        k1, k2 = kernel_sizes
        p1 = k1 // 2
        p2 = k2 // 2

        self.conv1 = nn.Conv2d(in_channels, filters[0], kernel_size=k1, padding=p1)
        self.conv2 = nn.Conv2d(filters[0], filters[1], kernel_size=k2, padding=p2)

        self.use_bn = use_bn
        self.bn1 = nn.BatchNorm2d(filters[0])
        self.bn2 = nn.BatchNorm2d(filters[1])

        if pool_type == "max":
            self.pool = nn.MaxPool2d(2, 2)
        else:
            self.pool = nn.AvgPool2d(2, 2)

        self.dropout = nn.Dropout(dropout)

        # После двух pooling размер уменьшается в 4 раза
        feature_size = img_size // 4
        self.fc1 = nn.Linear(filters[1] * feature_size * feature_size, 128)
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.conv1(x)
        if self.use_bn:
            x = self.bn1(x)
        x = torch.relu(x)
        x = self.pool(x)

        x = self.conv2(x)
        if self.use_bn:
            x = self.bn2(x)
        x = torch.relu(x)
        x = self.pool(x)

        x = x.view(x.size(0), -1)
        x = self.dropout(torch.relu(self.fc1(x)))
        x = self.fc2(x)
        return x

##### 0.3 Более глубокая CNN для 4-й части:

In [ ]:
class DeeperCNN(nn.Module):
    def __init__(self, in_channels=3, num_classes=10, img_size=32, dropout=0.3):
        super().__init__()

        self.conv1 = nn.Conv2d(in_channels, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)

        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)

        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)

        self.pool = nn.MaxPool2d(2, 2)
        self.dropout = nn.Dropout(dropout)

        feature_size = img_size // 8
        self.fc1 = nn.Linear(128 * feature_size * feature_size, 128)
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.pool(torch.relu(self.bn1(self.conv1(x))))
        x = self.pool(torch.relu(self.bn2(self.conv2(x))))
        x = self.pool(torch.relu(self.bn3(self.conv3(x))))

        x = x.view(x.size(0), -1)
        x = self.dropout(torch.relu(self.fc1(x)))
        x = self.fc2(x)
        return x